In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset

# 1. Download and format Split-MNIST
print("Downloading and processing MNIST dataset for CNN (1x28x28)...")
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)

# Keep the original 10-class dataset for pre-training the CNN
pretrain_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

tasks = []
pairs = [(0, 1), (2, 3), (4, 5), (6, 7), (8, 9)]

for (c0, c1) in pairs:
    train_mask = (train_dataset.targets == c0) | (train_dataset.targets == c1)
    X_train = train_dataset.data[train_mask].float() / 255.0
    X_train = (X_train - 0.1307) / 0.3081
    X_train = X_train.view(-1, 1, 28, 28)
    y_train = (train_dataset.targets[train_mask] == c1).float().view(-1, 1)
    
    test_mask = (test_dataset.targets == c0) | (test_dataset.targets == c1)
    X_test = test_dataset.data[test_mask].float() / 255.0
    X_test = (X_test - 0.1307) / 0.3081
    X_test = X_test.view(-1, 1, 28, 28)
    y_test = (test_dataset.targets[test_mask] == c1).float().view(-1, 1)
    
    name = f"Digits {c0} vs {c1}"
    tasks.append((name, X_train, y_train, X_test, y_test))

print(f"Created {len(tasks)} tasks.")



In [ ]:


class PyTorchDynamicNetwork(nn.Module):
    """
    Adjacency-matrix dynamic network with hard-freeze pathway separation.

    Architecture
    ────────────
    • W[max×max]          : weight matrix (single nn.Parameter)
    • M[max×max]          : structural mask (which connections exist)
    • trainable_mask[max×max] : gradient mask (which connections can learn)

    The gradient hook ensures:
        W.grad *= trainable_mask
    so frozen weights receive EXACTLY ZERO gradient — no optimizer state update,
    no momentum, nothing.  They are physically frozen.

    Lifecycle
    ─────────
    1. Network starts small (input + output + 4 hidden).
       All existing connections have trainable_mask = 1 → fully plastic.

    2. Train on Task 1.  Network learns.  Fisher (EMA of g²) builds up
       for weights that are actively used.

    3. When a new distribution arrives (detected by rising EWC stress):
       a. autonomous_freeze() sets trainable_mask = 0 for ALL current connections
          that have significant Fisher importance.
       b. grow_neurons() adds new neurons with trainable_mask = 1.
       c. New neurons connect to inputs AND to the output → they form a
          parallel pathway that can learn the new task without any gradient
          flowing through old frozen weights.

    4. Replay ensures the new pathway doesn't accidentally counteract the
       old pathway's contribution to the output.

    Why this works where EWC/SI/AGEM failed
    ────────────────────────────────────────
    Those methods tried to DISCOURAGE the optimizer from changing old weights
    (via penalties or gradient scaling).  But in a recurrent forward pass,
    the optimizer always found paths through unprotected weights.

    Hard-freeze doesn't discourage — it PREVENTS.  Zero gradient = zero change.
    The optimizer has no choice but to use the new neurons.
    """

    def __init__(self, input_dim: int, output_dim: int,
                 max_neurons: int = 200, steps: int = 3):
        super().__init__()
        self.input_dim   = input_dim
        self.output_dim  = output_dim
        self.max_neurons = max_neurons
        self.steps       = steps

        self.active_neurons = input_dim + output_dim + 4

        # ── Learnable parameters ──────────────────────────────────────────
        self.W = nn.Parameter(torch.randn(max_neurons, max_neurons) * 0.1)
        self.b = nn.Parameter(torch.zeros(max_neurons))

        # ── Structural mask (which connections exist) ─────────────────────
        self.register_buffer('M', torch.zeros(max_neurons, max_neurons))

        # ── Gradient mask (which connections can learn) ───────────────────
        # 1.0 = trainable,  0.0 = frozen (hard zero gradient)
        self.register_buffer('trainable_mask',
                             torch.ones(max_neurons, max_neurons))

        # ── Bias freeze mask ─────────────────────────────────────────────
        self.register_buffer('bias_trainable',
                             torch.ones(max_neurons))

        # ── Per-weight gradient conflict statistics ───────────────────────────
        #
        # We explicitly measure if the CURRENT task is fighting REPLAY memory.
        #
        # stress_num: EMA of (-g_task * g_replay).
        #             Positive if they pull in opposite directions.
        #
        # stress_den: EMA of (|g_task| * |g_replay|).
        #             Normalization factor.
        #
        # stress = stress_num / (stress_den + 1e-8)
        #
        #   stress ≈ -1.0  →  Consistent agreement (single task learning)
        #   stress ≈ 0.0   →  Independent noise (converged at minimum)
        #   stress > 0.0   →  True conflict (new task destroying old knowledge)
        #
        self.register_buffer('stress_num', torch.zeros(max_neurons, max_neurons))
        self.register_buffer('stress_den', torch.zeros(max_neurons, max_neurons))

        # ── Gradient hooks (registered once) ─────────────────────────────
        self._hook_registered = False

        self._init_random_connections()

    # ─────────────────────────────────────────────────────────────────────
    def _init_random_connections(self):
        for i in range(self.active_neurons):
            for j in range(self.active_neurons):
                if i != j and i >= self.input_dim:
                    if torch.rand(1).item() > 0.5:
                        self.M[i, j] = 1.0

    # ─────────────────────────────────────────────────────────────────────
    def _register_hooks(self):
        """Register gradient hooks that enforce the trainable_mask."""
        if self._hook_registered:
            return

        def _w_hook(grad):
            # Hard-zero gradients for frozen connections
            return grad * self.trainable_mask * self.M

        def _b_hook(grad):
            return grad * self.bias_trainable

        self.W.register_hook(_w_hook)
        self.b.register_hook(_b_hook)
        self._hook_registered = True

    # ─────────────────────────────────────────────────────────────────────
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Ensure hooks are registered on first forward
        if not self._hook_registered:
            self._register_hooks()

        batch = x.size(0)
        state = torch.zeros(batch, self.max_neurons, device=x.device)
        state[:, :self.input_dim] = x

        W_eff = self.W * self.M

        for _ in range(self.steps):
            new_state = torch.relu(torch.matmul(state, W_eff.T) + self.b)
            state = torch.cat([x, new_state[:, self.input_dim:]], dim=1)

        return state[:, self.input_dim: self.input_dim + self.output_dim]

    # ─────────────────────────────────────────────────────────────────────
    def update_fisher(self, beta: float = 0.999):
        """
        Update Fisher (EMA of grad²) as a diagnostic signal.
        Called every batch AFTER backward().
        """
        with torch.no_grad():
            if self.W.grad is not None:
                grad_sq = (self.W.grad ** 2) * self.M
                self.fisher.mul_(beta).add_((1 - beta) * grad_sq)

    # ─────────────────────────────────────────────────────────────────────
    def update_stress(self, g_task: torch.Tensor, g_replay: torch.Tensor, beta: float = 0.99):
        """
        Update explicit task-vs-replay conflict statistics.
        
        Args:
            g_task: Gradient of W computed on the current task's batch.
            g_replay: Gradient of W computed on a batch from the replay buffer.
        """
        with torch.no_grad():
            # Only consider active connections that are trainable
            active = self.M * self.trainable_mask
            g_t = (g_task * active).detach()
            g_r = (g_replay * active).detach()
            
            # The product is negative if they point in opposite directions.
            # We negate it so that conflict is POSITIVE.
            conflict = -(g_t * g_r)
            magnitude = g_t.abs() * g_r.abs()
            
            self.stress_num.mul_(beta).add_((1 - beta) * conflict)
            self.stress_den.mul_(beta).add_((1 - beta) * magnitude)

    # ─────────────────────────────────────────────────────────────────────
    def get_stress_matrix(self) -> torch.Tensor:
        """
        Per-weight conflict fraction in [-1.0, 1.0].
        
        Only active trainable connections have non-zero stress.
        """
        with torch.no_grad():
            stress = self.stress_num / (self.stress_den + 1e-8)
            # Clip safely
            stress = stress.clamp(min=-1.0, max=1.0)
            return stress * self.M * self.trainable_mask

    # ─────────────────────────────────────────────────────────────────────
    def get_network_stress(self, conflict_threshold: float = 0.3) -> float:
        """
        Fraction of active trainable connections that are in high conflict.
        
        Returns a value in [0, 1]:
          0.0  →  No connections are conflicted (stable or just noise)
          ~1.0 →  All trainable connections are conflicted
        """
        with torch.no_grad():
            stress = self.get_stress_matrix()
            active_trainable = (self.M * self.trainable_mask).bool()
            if active_trainable.sum() == 0:
                return 0.0

            high_conflict = (stress[active_trainable] > conflict_threshold).float()
            return high_conflict.mean().item()

    # ─────────────────────────────────────────────────────────────────────
    def stress_freeze(self, threshold: float = 0.3) -> int:
        """
        Autonomously freeze the most-conflicted trainable connections.
        
        Args:
            threshold: Any trainable connection with a stress > threshold
                       is permanently frozen.
        """
        with torch.no_grad():
            stress = self.get_stress_matrix()
            active_trainable = (self.M * self.trainable_mask).bool()
            if active_trainable.sum() == 0:
                return 0

            freeze_mask = (stress >= threshold) & active_trainable
            n_frozen = int(freeze_mask.sum().item())

            self.trainable_mask[freeze_mask] = 0.0

            # Also freeze biases of neurons whose incoming connections
            # are now mostly frozen (they are 'old' neurons)
            for i in range(self.input_dim, self.active_neurons):
                incoming = self.M[i, :self.active_neurons]
                if incoming.sum() == 0:
                    continue
                frozen_frac = (
                    (1 - self.trainable_mask[i, :self.active_neurons])
                    [incoming.bool()].mean()
                )
                if frozen_frac > 0.5:
                    self.bias_trainable[i] = 0.0

            return n_frozen

    # ─────────────────────────────────────────────────────────────────────
    def get_ewc_stress(self) -> float:
        """
        Compute how much "stress" the network is under.

        Stress = mean Fisher of currently-trainable connections.
        High stress = existing trainable weights are receiving large gradients
        = the current task is trying hard to change them = potential conflict.
        """
        active_trainable = self.trainable_mask * self.M
        if active_trainable.sum() == 0:
            return 0.0
        return (self.fisher * active_trainable).sum().item() / active_trainable.sum().item()

    # ─────────────────────────────────────────────────────────────────────
    def autonomous_freeze(self, fisher_threshold_percentile: float = 50.0):
        """
        AUTONOMOUSLY freeze connections that have high Fisher importance.

        This is NOT manual task-boundary freezing.  It is triggered by the
        training loop when stress exceeds a threshold.

        Process:
        1. Look at Fisher values for currently-trainable connections.
        2. Connections with Fisher above the percentile threshold → freeze
           (trainable_mask = 0).
        3. Connections below threshold → stay trainable (they weren't
           important, so they can be reused).

        After freezing:
        - The network CANNOT modify these connections anymore (gradient = 0).
        - New neurons must be grown to provide fresh pathway capacity.
        """
        with torch.no_grad():
            # Only consider currently-trainable + active connections
            active = (self.trainable_mask * self.M).bool()
            if active.sum() == 0:
                return 0

            fisher_vals = self.fisher[active]
            if fisher_vals.numel() == 0:
                return 0

            threshold = torch.quantile(fisher_vals, fisher_threshold_percentile / 100.0)

            # Freeze connections with Fisher >= threshold
            freeze_mask = (self.fisher >= threshold) & active
            n_frozen = freeze_mask.sum().item()

            self.trainable_mask[freeze_mask] = 0.0

            # Also freeze biases of neurons whose incoming connections are
            # mostly frozen (they are "old" neurons now)
            for i in range(self.input_dim, self.active_neurons):
                incoming = self.M[i, :self.active_neurons]
                if incoming.sum() == 0:
                    continue
                frozen_frac = (1 - self.trainable_mask[i, :self.active_neurons])[incoming.bool()].mean()
                if frozen_frac > 0.5:
                    self.bias_trainable[i] = 0.0

            return n_frozen

    # ─────────────────────────────────────────────────────────────────────
    def grow_neuron(self, num_connections: int = 5):
        """
        Activate next pre-allocated neuron.

        New neuron has:
          • trainable_mask = 1 for all its connections → fully plastic
          • Fisher = 0 → not considered important (yet)
          • Small random weights → doesn't disrupt existing output
          • Connects to BOTH inputs and outputs → forms a parallel pathway

        The output neuron connections to the new neuron are TRAINABLE,
        while output connections to old neurons remain FROZEN.
        """
        if self.active_neurons >= self.max_neurons:
            print("  [network] Max capacity reached.")
            return

        new_idx = self.active_neurons
        self.active_neurons += 1

        with torch.no_grad():
            # Clear the new neuron's slot
            self.W.data[new_idx, :] = 0.0
            self.W.data[:, new_idx] = 0.0
            self.b.data[new_idx]    = 0.0

            # New neuron is fully trainable
            self.trainable_mask[new_idx, :] = 1.0
            self.trainable_mask[:, new_idx] = 1.0
            self.bias_trainable[new_idx]    = 1.0

            # Reset stress buffers for the new neuron
            self.stress_num[new_idx, :] = 0.0
            self.stress_num[:, new_idx] = 0.0
            self.stress_den[new_idx, :] = 0.0
            self.stress_den[:, new_idx] = 0.0

        # Wire incoming: from inputs + existing hidden neurons
        sources = [s for s in range(new_idx)
                   if s < self.input_dim or
                   s >= self.input_dim + self.output_dim]
        if sources:
            # Prefer connecting to input neurons (direct fresh signal)
            input_sources = [s for s in sources if s < self.input_dim]
            hidden_sources = [s for s in sources if s >= self.input_dim + self.output_dim]

            # Always connect to all inputs
            for s in input_sources:
                self.M[new_idx, s] = 1.0
                with torch.no_grad():
                    self.W.data[new_idx, s] = torch.randn(1).item() * 0.1

            # Connect to a few random hidden neurons
            if hidden_sources:
                n_hidden_conn = min(num_connections, len(hidden_sources))
                chosen = np.random.choice(hidden_sources, n_hidden_conn, replace=False)
                for s in chosen:
                    self.M[new_idx, s] = 1.0
                    with torch.no_grad():
                        self.W.data[new_idx, s] = torch.randn(1).item() * 0.05

        # Wire outgoing: to output neurons AND a few hidden neurons
        output_indices = list(range(self.input_dim, self.input_dim + self.output_dim))
        for out_idx in output_indices:
            self.M[out_idx, new_idx] = 1.0
            self.trainable_mask[out_idx, new_idx] = 1.0  # explicitly trainable
            with torch.no_grad():
                self.W.data[out_idx, new_idx] = 0.0  # start at 0 — no disruption

        # Also connect to a few other hidden neurons
        hidden_targets = [t for t in range(self.input_dim + self.output_dim, new_idx)]
        if hidden_targets:
            n_out = min(num_connections, len(hidden_targets))
            chosen = np.random.choice(hidden_targets, n_out, replace=False)
            for t in chosen:
                self.M[t, new_idx] = 1.0
                with torch.no_grad():
                    self.W.data[t, new_idx] = torch.randn(1).item() * 0.05

    # ─────────────────────────────────────────────────────────────────────
    def n_frozen(self) -> int:
        """Number of frozen connections."""
        active = self.M.bool()
        return int((active & ~self.trainable_mask.bool()).sum().item())

    def n_trainable(self) -> int:
        """Number of trainable connections."""
        return int((self.M * self.trainable_mask).sum().item())



In [ ]:
class CNNFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2) # 14x14
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2) # 7x7
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(32 * 7 * 7, 64)
        self.relu3 = nn.ReLU()

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.flatten(x)
        x = self.relu3(self.fc(x))
        return x

class HybridModel(nn.Module):
    def __init__(self, cnn, device):
        super().__init__()
        self.cnn = cnn # Shared, frozen CNN
        self.dynamic = PyTorchDynamicNetwork(input_dim=64, output_dim=1, max_neurons=500).to(device)
        
    def forward(self, x):
        features = self.cnn(x)
        return self.dynamic(features)

class ReplayBuffer:
    def __init__(self, capacity: int = 500):
        self.capacity = capacity
        self.X = None
        self.y = None

    def add_data(self, X_new: torch.Tensor, y_new: torch.Tensor, num_samples: int = 100):
        idx = torch.randperm(X_new.size(0))[:num_samples]
        X_sub = X_new[idx].clone().detach()
        y_sub = y_new[idx].clone().detach()

        if self.X is None:
            self.X = X_sub
            self.y = y_sub
        else:
            self.X = torch.cat([self.X, X_sub], dim=0)
            self.y = torch.cat([self.y, y_sub], dim=0)

            if self.X.size(0) > self.capacity:
                keep = torch.randperm(self.X.size(0))[:self.capacity]
                self.X = self.X[keep]
                self.y = self.y[keep]

    def sample(self, batch_size: int = 64) -> tuple:
        if self.X is None:
            return None, None
        size = self.X.size(0)
        idx = torch.randint(0, size, (min(batch_size, size),))
        return self.X[idx], self.y[idx]

    def has_data(self) -> bool:
        return self.X is not None and self.X.size(0) > 0

def _reset_adam_for_neuron(optimizer: torch.optim.Adam, net: nn.Module, new_idx: int):
    if net.W in optimizer.state:
        s_W = optimizer.state[net.W]
        if 'exp_avg' in s_W:
            s_W['exp_avg'][new_idx, :] = 0.0
            s_W['exp_avg'][:, new_idx] = 0.0
            s_W['exp_avg_sq'][new_idx, :] = 0.0
            s_W['exp_avg_sq'][:, new_idx] = 0.0
            
    if hasattr(net, 'b') and net.b in optimizer.state:
        s_b = optimizer.state[net.b]
        if 'exp_avg' in s_b:
            s_b['exp_avg'][new_idx] = 0.0
            s_b['exp_avg_sq'][new_idx] = 0.0



In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running on {device}...")

print("\n" + "="*50)
print("--- Pre-Training CNN Feature Extractor ---")
print("="*50)

# Create CNN and temporary 10-class classifier
shared_cnn = CNNFeatureExtractor().to(device)
classifier = nn.Linear(64, 10).to(device)

optimizer_cnn = torch.optim.Adam(list(shared_cnn.parameters()) + list(classifier.parameters()), lr=1e-3)
criterion_cnn = nn.CrossEntropyLoss()

# Pre-train for 2 epochs on the standard MNIST 10-class task
shared_cnn.train()
classifier.train()
for epoch in range(2):
    total_loss = 0
    correct = 0
    total = 0
    for bx, by in pretrain_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer_cnn.zero_grad()
        
        features = shared_cnn(bx)
        out = classifier(features)
        
        loss = criterion_cnn(out, by)
        loss.backward()
        optimizer_cnn.step()
        
        total_loss += loss.item()
        preds = out.argmax(dim=1)
        correct += (preds == by).sum().item()
        total += by.size(0)
        
    print(f"Pre-Train Epoch {epoch+1}: Loss = {total_loss/len(pretrain_loader):.4f}, Acc = {100.0 * correct / total:.2f}%")

# FREEZE THE CNN!
for param in shared_cnn.parameters():
    param.requires_grad = False
shared_cnn.eval()
print("CNN Feature Extractor is now perfectly trained and permanently FROZEN.")



In [ ]:
class BaselineNet(nn.Module):
    def __init__(self, cnn):
        super().__init__()
        self.cnn = cnn # Frozen
        self.fc = nn.Sequential(
            nn.Linear(64, 100),
            nn.ReLU(),
            nn.Linear(100, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.fc(self.cnn(x))

print("\n" + "="*50)
print("--- Training Baseline (Frozen CNN + Static Network) ---")
print("="*50)

baseline_net = BaselineNet(shared_cnn).to(device)
# Only optimize the FC layers!
optimizer_b = torch.optim.Adam(baseline_net.fc.parameters(), lr=1e-3)
criterion_b = nn.BCELoss()

baseline_accuracies = {}

for task_id, (name, X_train, y_train, X_test, y_test) in enumerate(tasks):
    print(f"\nTraining Baseline on {name}...")
    
    dataset = TensorDataset(X_train, y_train)
    loader = DataLoader(dataset, batch_size=256, shuffle=True)
    
    for epoch in range(5): 
        baseline_net.train()
        for bx, by in loader:
            bx, by = bx.to(device), by.to(device)
            optimizer_b.zero_grad()
            loss = criterion_b(baseline_net(bx), by)
            loss.backward()
            optimizer_b.step()
            
    baseline_net.eval()
    print(f"Accuracy after {name}:")
    baseline_accuracies[name] = []
    with torch.no_grad():
        for eval_id in range(task_id + 1):
            t_name, _, _, t_X_test, t_y_test = tasks[eval_id]
            t_X_test_d = t_X_test.to(device)
            t_y_test_d = t_y_test.to(device)
            acc = ((baseline_net(t_X_test_d) > 0.5).float() == t_y_test_d).float().mean().item()
            baseline_accuracies[name].append((t_name, acc))
            print(f"  {t_name}: {acc*100:.1f}%")



In [ ]:
print("\n" + "="*50)
print("--- Training Our Hybrid (Frozen CNN + Dynamic Network) ---")
print("="*50)

hybrid_model = HybridModel(shared_cnn, device)
replay = ReplayBuffer(capacity=1000)

# ONLY optimize the Dynamic Network parameters!
optimizer = torch.optim.Adam(hybrid_model.dynamic.parameters(), lr=1e-3)
criterion = nn.MSELoss()

dynamic_accuracies = {}

for task_id, (name, X_train, y_train, X_test, y_test) in enumerate(tasks):
    print(f"\nTraining Hybrid Network on {name}...")
    
    dataset = TensorDataset(X_train, y_train)
    loader = DataLoader(dataset, batch_size=256, shuffle=True)
    
    loss_ema = 0.5
    stress_ema = 0.0
    batches_since_grow = 0
    
    for epoch in range(5):
        hybrid_model.train()
        # Ensure frozen CNN stays in eval mode (e.g. for BatchNorm if we had it)
        hybrid_model.cnn.eval() 
        
        for bx, by in loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            
            pred = hybrid_model(bx)
            loss = criterion(pred, by)
            
            if replay.has_data():
                rx, ry = replay.sample(128)
                rx, ry = rx.to(device), ry.to(device)
                r_pred = hybrid_model(rx)
                loss_r = criterion(r_pred, ry)
                
                # Compute gradient stress ONLY on the Dynamic Network's weights
                grads_task = torch.autograd.grad(loss, hybrid_model.dynamic.W, retain_graph=True, allow_unused=True)[0]
                grads_replay = torch.autograd.grad(loss_r, hybrid_model.dynamic.W, retain_graph=True, allow_unused=True)[0]
                
                if grads_task is not None and grads_replay is not None:
                    hybrid_model.dynamic.update_stress(grads_task, grads_replay)
                    
            loss.backward()
            optimizer.step()
            
            loss_ema = 0.9 * loss_ema + 0.1 * loss.item()
            stress_ema = hybrid_model.dynamic.get_network_stress()
            batches_since_grow += 1
            
            # Freeze condition
            if stress_ema > 0.3:
                n_frozen = hybrid_model.dynamic.stress_freeze(threshold=0.3)
                if n_frozen > 0:
                    hybrid_model.dynamic.grow_neuron(num_connections=15)
                    _reset_adam_for_neuron(optimizer, hybrid_model.dynamic, hybrid_model.dynamic.active_neurons - 1)
                    batches_since_grow = 0
                    print(f"  [Batch] FREEZE: {n_frozen} conns. Grew 1. active={hybrid_model.dynamic.active_neurons}")
            
            # Grow condition (loss)
            elif loss_ema > 0.2 and batches_since_grow > 20 and stress_ema <= 0.3:
                hybrid_model.dynamic.grow_neuron(num_connections=15)
                _reset_adam_for_neuron(optimizer, hybrid_model.dynamic, hybrid_model.dynamic.active_neurons - 1)
                batches_since_grow = 0
                print(f"  [Batch] GROW(Loss): active={hybrid_model.dynamic.active_neurons} loss={loss_ema:.4f}")
        
        print(f"  [Epoch {epoch+1:2d}] active={hybrid_model.dynamic.active_neurons} loss={loss_ema:.4f}")
        
    replay.add_data(X_train, y_train, num_samples=200)
    
    hybrid_model.eval()
    print(f"Accuracy after {name}:")
    dynamic_accuracies[name] = []
    with torch.no_grad():
        for eval_id in range(task_id + 1):
            t_name, _, _, t_X_test, t_y_test = tasks[eval_id]
            t_X_test_d = t_X_test.to(device)
            t_y_test_d = t_y_test.to(device)
            acc = ((hybrid_model(t_X_test_d) > 0.5).float() == t_y_test_d).float().mean().item()
            dynamic_accuracies[name].append((t_name, acc))
            print(f"  {t_name}: {acc*100:.1f}%")
    
    print(f"  Active Neurons: {hybrid_model.dynamic.active_neurons} (trainable={hybrid_model.dynamic.n_trainable()}, frozen={hybrid_model.dynamic.n_frozen()})")

